# Kathmandu PM2.5 Forecasting: Validation, Warning Evaluation, Failure Analysis & Artifact Export

This notebook covers the exact workflow steps:
1. **Feature Engineering & Chronological Train/Validation/Test Split**
2. **Validation Evaluation** (Persistence vs. Linear Regression MAE)
3. **Test Set Warning Evaluation** (Precision, Recall, True/Missed Warnings, False Alarms at 35 µg/m³)
4. **Failure Analysis** (Top 5 highest error test dates)
5. **Final Model Retraining (`train + validation`), Artifact Saving (`joblib` + `config.json`), & Final Test Evaluation**

In [ ]:
import json
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load daily dataset
df = pd.read_csv("PhoraDurbar_Kathmandu_PM25_Daily_Split_Baseline.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Feature engineering (leakage-safe)
df["lag_1"] = df["pm25_mean"].shift(1)
df["rolling_3"] = df["pm25_mean"].shift(1).rolling(window=3).mean()
df["month"] = df["date"].dt.month

model_df = df.dropna(subset=["pm25_mean", "lag_1", "rolling_3"]).copy()
model_df["target_date"] = model_df["date"]
model_df["target_pm25"] = model_df["pm25_mean"]

# Chronological split: Train, Validation, Test
test_start = pd.Timestamp("2023-08-08")
train_data = model_df[model_df["target_date"] < test_start].copy()
test = model_df[model_df["target_date"] >= test_start].copy()

val_split_idx = int(len(train_data) * 0.8)
train = train_data.iloc[:val_split_idx].copy()
validation = train_data.iloc[val_split_idx:].copy()

features = ["lag_1", "rolling_3", "month"]

# Train Linear Regression on train split
lr = LinearRegression()
lr.fit(train[features], train["target_pm25"])

validation_prediction = lr.predict(validation[features])
test_prediction = lr.predict(test[features])

print("Validation samples:", len(validation))
print("Validation persistence MAE:",
      mean_absolute_error(
          validation["target_pm25"],
          validation["lag_1"]
      ))

print("Validation Linear Regression MAE:",
      mean_absolute_error(
          validation["target_pm25"],
          validation_prediction
      ))

In [ ]:
# Evaluate warnings on the test set
threshold = 35

actual = test["target_pm25"].to_numpy()
model_pred = test_prediction
persistence_pred = test["lag_1"].to_numpy()

def warning_results(predictions, actual, threshold=35):
    actual_warning = actual > threshold
    predicted_warning = predictions > threshold

    true_warnings = (actual_warning & predicted_warning).sum()
    missed_warnings = (actual_warning & ~predicted_warning).sum()
    false_alarms = (~actual_warning & predicted_warning).sum()

    precision = (
        true_warnings / predicted_warning.sum()
        if predicted_warning.sum() else 0
    )
    recall = (
        true_warnings / actual_warning.sum()
        if actual_warning.sum() else 0
    )

    return {
        "actual_high_days": int(actual_warning.sum()),
        "predicted_warning_days": int(predicted_warning.sum()),
        "true_warnings": int(true_warnings),
        "missed_warnings": int(missed_warnings),
        "false_alarms": int(false_alarms),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
    }

print("MODEL WARNINGS")
print(warning_results(model_pred, actual))

print("PERSISTENCE WARNINGS")
print(warning_results(persistence_pred, actual))

In [ ]:
# Find failure examples
test_results = test[["target_date", "target_pm25"]].copy()
test_results["prediction"] = test_prediction
test_results["absolute_error"] = (
    test_results["target_pm25"] - test_results["prediction"]
).abs()

print(
    test_results
    .sort_values("absolute_error", ascending=False)
    .head(5)
)

In [ ]:
# Retrain the selected model using training plus validation data, then save it
final_train = pd.concat([train, validation]).sort_values("target_date")

final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

final_pipeline.fit(
    final_train[features],
    final_train["target_pm25"]
)

Path("artifacts").mkdir(exist_ok=True)

joblib.dump(final_pipeline, "artifacts/linear_regression_pm25.joblib")

with open("artifacts/config.json", "w") as file:
    json.dump(
        {
            "features": features,
            "threshold": 35,
            "model_version": "linear_regression_v1",
            "target": "next_day_daily_pm25",
            "station": "Phora Durbar, Kathmandu",
        },
        file,
        indent=2,
    )

print("Model and configuration saved.")

# Final test evaluation using final_pipeline
final_test_preds = final_pipeline.predict(test[features])
final_test_mae = mean_absolute_error(test["target_pm25"], final_test_preds)
final_test_rmse = np.sqrt(mean_squared_error(test["target_pm25"], final_test_preds))

print(f"Final Test MAE: {final_test_mae:.2f}")
print(f"Final Test RMSE: {final_test_rmse:.2f}")